# Heatmap Plotting (Intensity)
**Summary:** Evaluates intensity predictions from various Prosit models. 
Calculates spectral angles across different PTMs and plots heatmaps to compare model performance (e.g., Double Ac vs Double Gl).

**Required Files:**
- Predictions parquet files


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt
from glob import iglob
from typing import List
import re
import numpy as np
import math
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.pyplot as plt
import colorsys
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D


## Configuration
Paths and variables needed to run this notebook.


## Data Loading & Preprocessing
Load all prediction parquet files, filter for specific fragmentations, and compute median spectral angles.

In [ ]:
PROSIT_MODELS_GLOB = '<PATH>'
PLOT_OUTPUT_HEATMAP_1 = '<PATH_TO_PLOT_OUTPUT_HEATMAP_1>'
import os
os.makedirs(os.path.dirname(PLOT_OUTPUT_HEATMAP_1), exist_ok=True)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

test_ptms_files= [f for f in iglob(PROSIT_MODELS_GLOB)]

def filter_peptides_unmod(df, column):
    pattern = r"\[UNIMOD:(\d+)\]"
    
    def has_other_mod(seq):
        mods = re.findall(pattern, seq)
        # At least one modification present and at least one is not 35 or 4
        if mods and any(m not in ['35', '4'] for m in mods):
            return True
        return False
    
    return df[df[column].apply(~has_other_mod)]

df[~df['package'].str.contains('mod')]

In [ ]:
def filter_peptides_with_other_mods(df, column):
    pattern = r"\[UNIMOD:(\d+)\]"
    
    def has_other_mod(seq):
        mods = re.findall(pattern, seq)
        # At least one modification present and at least one is not 35 or 4
        if mods and any(m not in ['35', '4'] for m in mods):
            return True
        return False
    
    return df[df[column].apply(has_other_mod)]

In [ ]:
model_names = []
mods = []
sa_medians = []
for f in test_ptms_files:
  
    model_name =  f.split('/')[-2]
    mod_name =  f.split('/')[-1].replace('.parquet','')
    if mod_name not in unseen_packages:
        continue
    df = pd.read_parquet(f)
    df = df[df['fragmentation']=='HCD']
    #if 'basic' not in f and 'naive' not in f:
    
    if 'mono' in mod_name:
        df_sub= df[df['modified_sequence'].str.contains('K\[')]
        sa_median = round(df_sub['spectral_angle'].median(),2)
        model_names.append(model_name)
        mods.append('K(me)')
        sa_medians.append(sa_median)
        df_sub= df[df['modified_sequence'].str.contains('R\[')]
        sa_median = round(df_sub['spectral_angle'].median(),2)
        model_names.append(model_name)
        mods.append('R(me)')
        sa_medians.append(sa_median)
    elif 'OGal' in mod_name:
        df_sub= df[df['modified_sequence'].str.contains('S\[')]
        sa_median = round(df_sub['spectral_angle'].median(),2)
        model_names.append(model_name)
        mods.append('S(ga)')
        sa_medians.append(sa_median)
        df_sub= df[df['modified_sequence'].str.contains('T\[')]
        sa_median = round(df_sub['spectral_angle'].median(),2)
        model_names.append(model_name)
        mods.append('T(ga)')
        sa_medians.append(sa_median)
    elif 'OGlc' in mod_name:
        print('glc')
        df_sub= df[df['modified_sequence'].str.contains('S\[')]
        sa_median = round(df_sub['spectral_angle'].median(),2)
        model_names.append(model_name)
        mods.append('S(gl)')
        sa_medians.append(sa_median)
        df_sub= df[df['modified_sequence'].str.contains('T\[')]
        sa_median = round(df_sub['spectral_angle'].median(),2)
        model_names.append(model_name)
        mods.append('T(gl)')
        sa_medians.append(sa_median)
    else:
        sa_median = round(df['spectral_angle'].median(),2)
        model_names.append(model_name)
        mods.append(mod_name)
        sa_medians.append(sa_median)

    

all_train_data_df = pd.DataFrame()
all_train_data_df['model_names'] = model_names
all_train_data_df['mods'] = mods
all_train_data_df['sa_median'] = sa_medians




## Heatmap Visualization
Pivot the computed medians and plot heatmaps comparing the different models.

In [ ]:
pt = pd.pivot_table(all_train_data_df, 
                    index='model_names', values=['sa_median'], columns='mods')

pt.columns = pt.columns.get_level_values(1)
pt.index = pd.CategoricalIndex(pt.index, categories= ['double_ac','double_gl'])

pt.sort_index(level=0, inplace=True)





fig, ax = plt.subplots(figsize = (16, 6))
ax.set_title("Seen Mods Spectral Angle")

bounds = np.concatenate([
    np.linspace(0.5, 0.7, 20),
    np.linspace(0.7, 0.8, 50)[1:],      # e.g. [0.0, 0.233, 0.467, 0.7]
    np.linspace(0.8, 0.9, 80)[1:],
    np.linspace(0.9, 0.93, 60)[1:]     # e.g. [0.743, 0.786, 0.829, 0.871, 0.914, 0.957, 1.0]
])
# Total number of colors needed = len(bounds) - 1
n_bins = len(bounds) - 1

# Discretize the colormap to n_bins colors
base_cmap = plt.get_cmap('YlOrRd')
discrete_cmap = ListedColormap(base_cmap(np.linspace(0, 1, n_bins)))

# Create the norm
norm = BoundaryNorm(boundaries=bounds, ncolors=300)
ticks_to_show = [0.5, 0.75, 0.8, 0.9, 0.93]
sns.heatmap(pt, cmap="mako", annot=True, norm=norm, cbar_kws={'ticks': ticks_to_show, 'format': '%.2f'}, xticklabels=1, yticklabels=1,vmin=0.12,vmax=0.93)
plt.yticks(rotation=0)
plt.tight_layout()
